# Academic Advisor: model training (CSC 303)

This notebook trains the models behind the Academic Advisor web app. It runs end to end on
Google Colab (**Runtime > Run all**) and produces **`advisor_model.pkl`**, which you place in the
`models/` folder of the repo and serve with Flask.

**Pipeline:** synthetic student records -> exploratory look -> compare candidate models with
cross-validation -> train the final classifier (standing) and regressor (semester GPA) ->
evaluate on a hold-out set -> save the `.pkl` bundle -> download.

**What the models learn from:** academic level, CGPA, last semester GPA, average course score,
credit load, average difficulty and confidence, study hours and days, attendance, assignment
habits, repeat courses and the gap to the target CGPA. Everything else the student enters
(study method, challenges, exam date...) drives the rule-based part of the advisor instead.

> The data is synthetic (generated from plausible rules plus noise). It demonstrates the
> workflow; for real use, replace it with anonymised records from your department.

## 1. Setup

In [ ]:
import sys, sklearn, pandas as pd, numpy as np
print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

In [ ]:
!mkdir -p advisor models data
!touch advisor/__init__.py

## 2. Project code
These two cells write the same `advisor/data.py` and `advisor/model.py` used by the web app.

In [ ]:
%%writefile advisor/data.py
"""Synthetic student-record generator and shared domain constants.

No real student records are used anywhere in this project. The generator encodes
common-sense relationships (attendance, study time, confidence, difficulty, carry-overs,
workload, assignment habits) so that a model has something realistic to learn.
Swap `generate_dataset()` for a loader of real, anonymised records when available.

The model only ever sees the numeric FEATURES below, in this order. Everything else the
student types (department, study method, challenges, exam date...) drives the rule-based
part of the advisor instead.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

# Features the ML model is trained on (order matters and is stored in the .pkl).
FEATURES = [
    "level",  # 1..5 (100L..500L)
    "cgpa",
    "last_gpa",
    "avg_score",  # credit-weighted average course score, %
    "credit_load",  # total credit units this semester
    "avg_difficulty",  # 1 easy .. 3 difficult (credit-weighted)
    "avg_confidence",  # 1 low .. 5 high (credit-weighted)
    "study_hours_day",
    "study_days_week",
    "attendance_pct",
    "assignment_rate",  # 0.9 usually, 0.6 sometimes, 0.3 rarely
    "carryovers",  # courses being repeated
    "goal_gap",  # max(0, target CGPA - current CGPA)
]

TARGET_CLASS = "next_standing"
TARGET_GPA = "next_gpa"
# Kept for older callers that only need the class label column name.
TARGET = TARGET_CLASS

# Ordered from worst to best.
CLASSES = ["At Risk", "Fair", "Good", "Excellent"]

DIFFICULTY = {"easy": 1, "moderate": 2, "difficult": 3}
ASSIGNMENT_RATE = {"usually": 0.9, "sometimes": 0.6, "rarely": 0.3}

# Nigerian 5-point scale (lower bound, label). Check your department's handbook.
DEGREE_CLASSES = [
    (4.50, "First Class"),
    (3.50, "Second Class Upper"),
    (2.40, "Second Class Lower"),
    (1.50, "Third Class"),
    (1.00, "Pass"),
]


def degree_class(cgpa: float) -> str:
    """Return the degree class for a CGPA on the 5-point scale."""
    for lower, label in DEGREE_CLASSES:
        if cgpa >= lower:
            return label
    return "Below Pass"


def gpa_to_standing(gpa: float) -> str:
    """Map a semester GPA to the standing label the model predicts."""
    if gpa >= 4.5:
        return "Excellent"
    if gpa >= 3.5:
        return "Good"
    if gpa >= 2.4:
        return "Fair"
    return "At Risk"


def estimate_score_from_gpa(gpa: float) -> float:
    """Typical average course score (%) for a GPA. Used when a student enters no scores."""
    return float(np.clip(30 + 8.5 * gpa, 20, 100))


def generate_dataset(n: int = 4000, seed: int = 42) -> pd.DataFrame:
    """Create `n` synthetic student records with next-semester GPA and standing."""
    rng = np.random.default_rng(seed)

    ability = rng.normal(0.0, 1.0, n)  # hidden driver shared by several features

    level = rng.integers(1, 6, n)
    cgpa = np.clip(3.4 + 0.7 * ability + rng.normal(0, 0.25, n), 1.0, 5.0)
    last_gpa = np.clip(cgpa + rng.normal(0, 0.45, n), 0.0, 5.0)
    avg_score = np.clip(30 + 8.5 * last_gpa + rng.normal(0, 6, n), 20, 100)
    credit_load = rng.choice([15, 18, 21, 24, 27], size=n, p=[0.10, 0.30, 0.30, 0.22, 0.08])
    avg_difficulty = np.clip(2.0 - 0.25 * ability + rng.normal(0, 0.4, n), 1, 3)
    avg_confidence = np.clip(3.0 + 0.6 * ability + rng.normal(0, 0.7, n), 1, 5)
    study_hours_day = np.clip(2.2 + 0.5 * ability + rng.normal(0, 1.0, n), 0, 10)
    study_days_week = np.clip(np.round(4.5 + 0.6 * ability + rng.normal(0, 1.2, n)), 1, 7)
    attendance = np.clip(78 + 5 * ability + rng.normal(0, 11, n), 30, 100)
    habit = 0.5 * ability + rng.normal(0, 0.8, n)
    assignment_rate = np.where(habit > 0.1, 0.9, np.where(habit > -0.8, 0.6, 0.3))
    carryovers = np.clip(rng.poisson(np.clip(1.0 - 0.8 * ability, 0.05, None)), 0, 6)
    target = np.clip(cgpa + np.abs(rng.normal(0.3, 0.4, n)), 1.0, 5.0)
    goal_gap = target - cgpa

    weekly_study = study_hours_day * study_days_week

    # Hidden "true" process for next-semester GPA, plus noise.
    next_gpa = (
        0.30
        + 0.55 * cgpa
        + 0.25 * last_gpa
        + 0.010 * (avg_score - 60)
        + 0.012 * (attendance - 75)
        + 0.025 * (weekly_study - 10)
        + 0.10 * (avg_confidence - 3)
        - 0.12 * (avg_difficulty - 2)
        + 0.25 * (assignment_rate - 0.6)
        - 0.12 * carryovers
        - 0.040 * np.maximum(credit_load - 21, 0)
        + rng.normal(0, 0.30, n)
    )
    next_gpa = np.round(np.clip(next_gpa, 0.0, 5.0), 2)  # round first so label and GPA agree

    df = pd.DataFrame(
        {
            "level": level.astype(int),
            "cgpa": cgpa.round(2),
            "last_gpa": last_gpa.round(2),
            "avg_score": avg_score.round(1),
            "credit_load": credit_load.astype(int),
            "avg_difficulty": avg_difficulty.round(2),
            "avg_confidence": avg_confidence.round(2),
            "study_hours_day": study_hours_day.round(1),
            "study_days_week": study_days_week.astype(int),
            "attendance_pct": attendance.round(0),
            "assignment_rate": assignment_rate,
            "carryovers": carryovers.astype(int),
            "goal_gap": goal_gap.round(2),
        }
    )
    df[TARGET_GPA] = next_gpa
    df[TARGET_CLASS] = [gpa_to_standing(g) for g in next_gpa]
    return df


if __name__ == "__main__":
    out = generate_dataset()
    out.to_csv("data/students.csv", index=False)
    print(out[TARGET_CLASS].value_counts())
    print(f"Wrote {len(out)} rows to data/students.csv")

In [ ]:
%%writefile advisor/model.py
"""Training, evaluation and (de)serialisation of the advisor models.

The artefact written to disk (`models/advisor_model.pkl`) is a dict "bundle":

    {
        "model":           fitted classifier (has predict_proba) -> standing,
        "regressor":       fitted regressor -> expected semester GPA,
        "feature_names":   list[str]  - column order both models expect,
        "classes":         list[str]  - label order of predict_proba,
        "metrics":         dict       - hold-out accuracy, macro-F1, MAE, R2,
        "sklearn_version": str        - version used for training,
    }

The Colab notebook writes the same structure, so either source of the .pkl works.
"""

from __future__ import annotations

import joblib
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

from advisor.data import FEATURES, TARGET_CLASS, TARGET_GPA

DEFAULT_MODEL_PATH = "models/advisor_model.pkl"


def build_model(seed: int = 42) -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=8,
        class_weight="balanced",
        random_state=seed,
        n_jobs=-1,
    )


def build_regressor(seed: int = 42) -> RandomForestRegressor:
    return RandomForestRegressor(
        n_estimators=150,
        min_samples_leaf=8,
        random_state=seed,
        n_jobs=-1,
    )


def split(df: pd.DataFrame, seed: int = 42):
    """The one train/test split used everywhere (also in the Colab notebook)."""
    return train_test_split(
        df[FEATURES],
        df[[TARGET_CLASS, TARGET_GPA]],
        test_size=0.2,
        stratify=df[TARGET_CLASS],
        random_state=seed,
    )


def train_bundle(df: pd.DataFrame, seed: int = 42) -> dict:
    """Fit both models on `df`, evaluate on a stratified hold-out split, return a bundle."""
    X_train, X_test, y_train, y_test = split(df, seed)
    clf = build_model(seed).fit(X_train, y_train[TARGET_CLASS])
    reg = build_regressor(seed).fit(X_train, y_train[TARGET_GPA])

    pred = clf.predict(X_test)
    gpa_pred = reg.predict(X_test)
    metrics = {
        "accuracy": round(float(accuracy_score(y_test[TARGET_CLASS], pred)), 4),
        "macro_f1": round(float(f1_score(y_test[TARGET_CLASS], pred, average="macro")), 4),
        "gpa_mae": round(float(mean_absolute_error(y_test[TARGET_GPA], gpa_pred)), 4),
        "gpa_r2": round(float(r2_score(y_test[TARGET_GPA], gpa_pred)), 4),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
    }
    return {
        "model": clf,
        "regressor": reg,
        "feature_names": list(FEATURES),
        "classes": [str(c) for c in clf.classes_],
        "metrics": metrics,
        "sklearn_version": sklearn.__version__,
    }


def save_bundle(bundle: dict, path: str = DEFAULT_MODEL_PATH) -> None:
    joblib.dump(bundle, path, compress=3)


def load_bundle(path: str = DEFAULT_MODEL_PATH) -> dict:
    bundle = joblib.load(path)
    missing = {"model", "feature_names", "classes"} - set(bundle)
    if missing:
        raise ValueError(f"{path} is not a valid advisor bundle (missing {sorted(missing)})")
    return bundle

## 3. Generate and inspect the dataset

In [ ]:
import matplotlib.pyplot as plt
from advisor.data import FEATURES, TARGET_CLASS, TARGET_GPA, CLASSES, generate_dataset

df = generate_dataset(n=4000, seed=42)
df.to_csv("data/students.csv", index=False)
print(df.shape)
df.head()

In [ ]:
print(df[TARGET_CLASS].value_counts().reindex(CLASSES))
df[FEATURES + [TARGET_GPA]].describe().round(2).T[["mean", "std", "min", "max"]]

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
df[TARGET_CLASS].value_counts().reindex(CLASSES).plot.bar(ax=ax[0], color="#d6283c")
ax[0].set_title("Next-semester standing (class target)")
ax[0].set_xlabel(""); ax[0].tick_params(axis="x", rotation=0)

ax[1].hist(df[TARGET_GPA], bins=30, color="#0f172a")
ax[1].set_title("Next-semester GPA (regression target)"); ax[1].set_xlabel("GPA")

corr = df[FEATURES + [TARGET_GPA]].corr()
im = ax[2].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax[2].set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=7)
ax[2].set_yticks(range(len(corr)), corr.columns, fontsize=7)
ax[2].set_title("Correlations")
fig.colorbar(im, ax=ax[2], fraction=0.046)
plt.tight_layout(); plt.show()

## 4. Compare candidate models (5-fold cross-validation)

In [ ]:
from sklearn.ensemble import (GradientBoostingClassifier, GradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from advisor.model import build_model, build_regressor, save_bundle, split, train_bundle

# The same split the final model uses (and the web app's training code uses).
X_train, X_test, y_train, y_test = split(df, seed=42)

classifiers = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced")),
    "Random Forest (chosen)": build_model(),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}
rows = []
for name, model in classifiers.items():
    s = cross_val_score(model, X_train, y_train[TARGET_CLASS],
                        cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1_macro")
    rows.append({"task": "standing (classifier)", "model": name, "score": round(s.mean(), 4), "metric": "macro-F1"})

regressors = {
    "Ridge Regression": make_pipeline(StandardScaler(), Ridge()),
    "Random Forest (chosen)": build_regressor(),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}
for name, model in regressors.items():
    s = cross_val_score(model, X_train, y_train[TARGET_GPA],
                        cv=KFold(5, shuffle=True, random_state=42), scoring="neg_mean_absolute_error")
    rows.append({"task": "semester GPA (regressor)", "model": name, "score": round(-s.mean(), 4), "metric": "MAE (lower is better)"})
pd.DataFrame(rows)

The scores are close. The **Random Forest** is used for both tasks: it captures non-linear
interactions (for example, heavy credit load only hurts when study time is low), needs no feature
scaling, exposes feature importances for the app, and stays small once compressed. The classifier
gives the standing label and its probabilities; the regressor gives the expected semester GPA that
feeds the CGPA projection in the app.

## 5. Train the final models and evaluate on the hold-out set

In [ ]:
import sklearn
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

bundle = train_bundle(df, seed=42)
model, regressor = bundle["model"], bundle["regressor"]
pred = model.predict(X_test)
gpa_pred = regressor.predict(X_test)

print(classification_report(y_test[TARGET_CLASS], pred, labels=CLASSES))
print({k: v for k, v in bundle["metrics"].items()})

fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
ConfusionMatrixDisplay.from_predictions(y_test[TARGET_CLASS], pred, labels=CLASSES, ax=ax[0],
                                        cmap="Blues", colorbar=False)
ax[0].set_title("Standing: confusion matrix (hold-out)")

ax[1].scatter(y_test[TARGET_GPA], gpa_pred, s=8, alpha=0.5, color="#d6283c")
ax[1].plot([0, 5], [0, 5], color="#0f172a", lw=1)
ax[1].set_xlabel("Actual GPA"); ax[1].set_ylabel("Predicted GPA"); ax[1].set_title("Semester GPA: predicted vs actual")

imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
imp.plot.barh(ax=ax[2], color="#0f172a"); ax[2].set_title("What the classifier relies on")
plt.tight_layout(); plt.show()

## 6. Save the `.pkl` bundle and verify it reloads

In [ ]:
import joblib

save_bundle(bundle, "models/advisor_model.pkl")

# Sanity check: reload from disk and predict for one hypothetical student.
loaded = joblib.load("models/advisor_model.pkl")
student = pd.DataFrame([{
    "level": 3, "cgpa": 3.6, "last_gpa": 3.4, "avg_score": 63, "credit_load": 18,
    "avg_difficulty": 2.4, "avg_confidence": 2.8, "study_hours_day": 3, "study_days_week": 5,
    "attendance_pct": 82, "assignment_rate": 0.6, "carryovers": 1, "goal_gap": 0.2,
}])[loaded["feature_names"]]
probs = loaded["model"].predict_proba(student)[0]
print({c: round(float(p), 3) for c, p in zip(loaded["classes"], probs)})
print("Expected semester GPA:", round(float(loaded["regressor"].predict(student)[0]), 2))
print("Saved models/advisor_model.pkl", loaded["metrics"])

## 7. Download the model

In [ ]:
try:
    from google.colab import files
    files.download("models/advisor_model.pkl")
except ImportError:
    print("Not running on Colab; the file is at models/advisor_model.pkl")

**Next:** copy the downloaded `advisor_model.pkl` into the repo's `models/` folder (replace the old
one), run `python app.py`, and test it in the browser. If loading fails with a scikit-learn version
error, the app retrains a fresh model automatically; or install the version printed in section 1
(`pip install scikit-learn==<version>`) to use the Colab file as is.